In [1]:
"""
Weight Sharing — HashedNets
============================
HashedNets (Chen et al. 2015) compresses neural networks by grouping weights
into shared hash buckets. Each weight parameter's index is mapped to a bucket
via a hash function; all weights in the same bucket share one value.

METHOD
======
  For each Conv2d / Linear layer:
    - All weight indices (c_out, c_in, kH, kW) are hashed → bucket_id in [0, K)
    - K = int(total_params * compression_ratio)  buckets per layer
    - Forward pass rebuilds the full weight tensor by indexing the shared values
    - Gradient w.r.t. bucket values is accumulated from all mapped indices

  This is an approximation technique: no decomposition, no matrix factorization.
  Accuracy recovers with fine-tuning; compression scales with bucket_fraction.

MATH
====
  Given weight tensor W ∈ R^n, hash function h: [n] → [K]
    W[i] = shared_values[h(i)]    for all i

  K = ceil(n * bucket_fraction)   →  K/n fraction of original parameters

  Hash used: xxhash-style polynomial:  h(i) = ((seed * i) XOR i) mod K
  (Implemented in pure PyTorch, no external hash library needed.)

OUTPUT
======
  __2__HashedNets.json
"""

import os, json, time, copy, tempfile, warnings, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# for reproducibility
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[init] PyTorch  : {torch.__version__}", flush=True)
print(f"[init] Device   : {DEVICE}", flush=True)

# ── Config ────────────────────────────────────────────────────────────────────
ORIGINAL_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
MODEL_ARCH            = "resnet50"
OUTPUT_JSON           = "__2__HashedNets.json"

NUM_CLASSES  = 10
BATCH_SIZE   = 128 if DEVICE.type == "cuda" else 64
NUM_WORKERS  = 0
PIN_MEMORY   = DEVICE.type == "cuda"

# Fraction of parameters to keep as unique shared buckets.
# 0.25 → 75% parameter reduction within each layer.
BUCKET_FRACTION = 0.25

# Optional fine-tuning (set 0 to skip)
FINETUNE_EPOCHS = 5
FINETUNE_LR     = 1e-3

# Hash seed (arbitrary prime for mixing)
HASH_SEED = 2654435761  # Knuth multiplicative hash constant

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"[init] BucketFraction: {BUCKET_FRACTION}  FineTuneEpochs: {FINETUNE_EPOCHS}", flush=True)

[init] PyTorch  : 2.12.0.dev20260324+cu128
[init] Device   : cuda
[init] BucketFraction: 0.25  FineTuneEpochs: 5


In [2]:
# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(arch="resnet50", num_classes=10):
    m = models.resnet50(weights=None) if arch == "resnet50" else models.resnet18(weights=None)
    m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(m.fc.in_features, num_classes)
    return m


def load_model(path, arch):
    print(f"[model] Loading {arch} from {path} ...", flush=True)
    m = build_model(arch, NUM_CLASSES)
    m.load_state_dict(torch.load(path, map_location="cpu"))
    m = m.to(DEVICE).eval()
    print(f"[model] Loaded. Params: {sum(p.numel() for p in m.parameters()):,}", flush=True)
    return m

In [3]:
# ── Data ──────────────────────────────────────────────────────────────────────
def get_loaders():
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    train_ds = torchvision.datasets.CIFAR10("../data", train=True,  download=True, transform=train_tf)
    test_ds  = torchvision.datasets.CIFAR10("../data", train=False, download=True, transform=test_tf)
    train_ld = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_ld  = DataLoader(test_ds,  512,        shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    return train_ld, test_ld

In [4]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def model_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        return os.path.getsize(tmp) / 1024**2
    finally:
        os.remove(tmp)


def param_count(model):
    return sum(p.numel() for p in model.parameters())


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        preds.extend(model(x).argmax(1).cpu().numpy())
        labels.extend(y.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


def measure_inference(model: nn.Module, device, batch_size=128, runs=50) -> dict:
    """
    Returns a dict with per-image (single), batch, and throughput timings.
    Uses CUDA events on GPU for highest accuracy; falls back to perf_counter on CPU.
    """
    m = model.eval().to(device)
    use_cuda = device.type == "cuda"

    def _timed_run(inp: torch.Tensor, n: int) -> float:
        """Returns total elapsed seconds for n forward passes."""
        with torch.no_grad():
            # warm-up
            for _ in range(3):
                m(inp)
            if use_cuda:
                torch.cuda.synchronize()
                start_evt = torch.cuda.Event(enable_timing=True)
                end_evt   = torch.cuda.Event(enable_timing=True)
                start_evt.record()
                for _ in range(n):
                    m(inp)
                end_evt.record()
                torch.cuda.synchronize()
                return start_evt.elapsed_time(end_evt) / 1000.0  # ms → s
            else:
                t0 = time.perf_counter()
                for _ in range(n):
                    m(inp)
                return time.perf_counter() - t0

    # Single-image latency
    single = torch.randn(1, 3, 32, 32, device=device)
    single_s = _timed_run(single, runs)
    single_ms = single_s / runs * 1000

    # Batch latency
    batch = torch.randn(batch_size, 3, 32, 32, device=device)
    batch_s = _timed_run(batch, runs)
    batch_ms = batch_s / runs * 1000

    per_img_ms  = batch_ms / batch_size
    throughput  = batch_size * runs / batch_s   # images / second

    timing_method = (
        "CUDA events + torch.cuda.synchronize()" if use_cuda
        else "time.perf_counter() (CPU)"
    )

    return {
        "single_img_gpu_ms"  : round(single_ms, 4),
        "batch128_gpu_ms"    : round(batch_ms,  4),
        "per_img_gpu_ms"     : round(per_img_ms, 6),
        "throughput_imgs_sec": round(throughput, 2),
        "timing_method"      : timing_method,
    }


def compute_flops(model: nn.Module, device, input_size=(1, 3, 32, 32)) -> float:
    """
    Estimates FLOPs (as GFLOPs) by registering forward hooks on Conv2d and Linear layers.
    Counts multiply-accumulate operations × 2 (one mul + one add = 2 FLOPs).
    Returns GFLOPs (float).
    """
    m = model.eval().to(device)
    total_flops = [0]
    hooks = []

    def conv_hook(module, inp, out):
        # out: (N, C_out, H_out, W_out)
        N, C_out, H_out, W_out = out.shape
        C_in  = module.in_channels
        kH, kW = module.kernel_size if isinstance(module.kernel_size, tuple) else (module.kernel_size, module.kernel_size)
        groups = module.groups
        # MACs per output element = (C_in / groups) * kH * kW
        macs = N * C_out * H_out * W_out * (C_in // groups) * kH * kW
        total_flops[0] += 2 * macs

    def linear_hook(module, inp, out):
        # out: (N, C_out)
        in_f  = module.in_features
        out_f = module.out_features
        N     = inp[0].shape[0]
        total_flops[0] += 2 * N * in_f * out_f

    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            hooks.append(mod.register_forward_hook(conv_hook))
        elif isinstance(mod, nn.Linear):
            hooks.append(mod.register_forward_hook(linear_hook))

    dummy = torch.randn(*input_size, device=device)
    with torch.no_grad():
        m(dummy)

    for h in hooks:
        h.remove()

    return round(total_flops[0] / 1e9, 6)  # GFLOPs



In [5]:
# ── HashedNets Core ───────────────────────────────────────────────────────────
def build_hash_indices(num_weights: int, num_buckets: int, seed: int = HASH_SEED) -> torch.Tensor:
    """
    Pure-PyTorch Knuth multiplicative hash:
      h(i) = ((seed * i) XOR i) mod K

    Returns LongTensor of shape (num_weights,) on CPU with values in [0, num_buckets).
    """
    idx    = torch.arange(num_weights, dtype=torch.long)
    hashed = (seed * idx) ^ idx
    hashed = hashed % num_buckets
    return hashed.abs()   # always CPU

class HashedLayer(nn.Module):
    """
    Wraps a Conv2d or Linear layer with HashedNets weight sharing.

    The original weight tensor W ∈ R^n is replaced by:
      - shared_values: nn.Parameter of shape (K,)   where K = ceil(n * bucket_fraction)
      - hash_indices : LongTensor of shape (n,)      mapping each weight to a bucket

    Forward:
      W_reconstructed = shared_values[hash_indices].reshape(original_shape)
      output = F.conv2d(x, W_reconstructed, ...) or F.linear(x, W_reconstructed, ...)

    Gradient flow:
      Autograd naturally accumulates gradients from all indices mapped to the same bucket.

    FIX: all scatter operations are performed on CPU; shared_values / hash_indices are
         moved to the target device only when the module is moved via .to() / .cuda().
    """
    def __init__(self, original_layer: nn.Module, bucket_fraction: float):
        super().__init__()
        assert isinstance(original_layer, (nn.Conv2d, nn.Linear)), \
            f"HashedLayer only wraps Conv2d or Linear, got {type(original_layer)}"

        self.is_conv      = isinstance(original_layer, nn.Conv2d)
        W                 = original_layer.weight.data
        self.weight_shape = W.shape
        n = W.numel()
        K = max(1, math.ceil(n * bucket_fraction))

        # ── Build hash indices (always on CPU) ────────────────────────────────
        hash_idx = build_hash_indices(n, K)   # CPU LongTensor
        self.register_buffer("hash_indices", hash_idx)

        # ── Initialise shared values by averaging weights per bucket ──────────
        # IMPORTANT: keep everything on CPU here; scatter_add_ requires
        # all tensors on the same device.  The parameter will be moved to
        # the target device later when the parent model calls .to(device).
        flat_W      = W.reshape(-1).cpu()          # ← ensure CPU
        shared_vals = torch.zeros(K, dtype=W.dtype)          # CPU
        counts      = torch.zeros(K, dtype=torch.long)       # CPU

        shared_vals.scatter_add_(0, hash_idx, flat_W)
        counts.scatter_add_(0, hash_idx, torch.ones(n, dtype=torch.long))
        counts.clamp_(min=1)
        shared_vals = shared_vals / counts.float()

        self.shared_values = nn.Parameter(shared_vals)       # CPU until .to(device)

        # ── Bias (not hashed — tiny tensor, keep as-is) ───────────────────────
        bias       = original_layer.bias
        self.bias  = nn.Parameter(bias.clone()) if bias is not None else None

        # ── Conv / Linear metadata ────────────────────────────────────────────
        if self.is_conv:
            self.stride   = original_layer.stride
            self.padding  = original_layer.padding
            self.dilation = original_layer.dilation
            self.groups   = original_layer.groups
        else:
            self.in_features  = original_layer.in_features
            self.out_features = original_layer.out_features

    def _reconstruct_weight(self):
        """Rebuild full weight tensor from shared values via hash lookup."""
        return self.shared_values[self.hash_indices].reshape(self.weight_shape)

    def forward(self, x):
        W = self._reconstruct_weight()
        if self.is_conv:
            return F.conv2d(x, W, self.bias,
                            stride=self.stride, padding=self.padding,
                            dilation=self.dilation, groups=self.groups)
        else:
            return F.linear(x, W, self.bias)

    def extra_repr(self):
        K = self.shared_values.shape[0]
        n = self.hash_indices.shape[0]
        return f"buckets={K}, weights={n}, ratio={K/n:.3f}"

# ── Apply HashedNets ──────────────────────────────────────────────────────────
def count_hashed_params(model: nn.Module) -> int:
    """Count effective unique parameters after hashing (buckets + biases + BN)."""
    total = 0
    for m in model.modules():
        if isinstance(m, HashedLayer):
            total += m.shared_values.numel()
            if m.bias is not None:
                total += m.bias.numel()
        elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            for p in m.parameters():
                total += p.numel()
    return total

def apply_hashednets(model: nn.Module, bucket_fraction: float) -> nn.Module:
    """
    Replace all Conv2d and Linear layers with HashedLayer.
    Skips layers where hashing would not reduce parameter count.
    The model is moved to DEVICE *after* all layers are replaced so that
    .to(DEVICE) propagates through HashedLayer buffers/parameters correctly.
    """
    model = copy.deepcopy(model).cpu()   # work entirely on CPU during replacement

    def _replace(parent: nn.Module):
        for name, child in list(parent.named_children()):
            if isinstance(child, (nn.Conv2d, nn.Linear)):
                n = child.weight.numel()
                K = max(1, math.ceil(n * bucket_fraction))
                if K < n:
                    print(f"    Hash: {type(child).__name__} {tuple(child.weight.shape)} "
                          f"→ {K}/{n} buckets ({K/n*100:.1f}%)", flush=True)
                    setattr(parent, name, HashedLayer(child, bucket_fraction))
                else:
                    print(f"    Skip (too small): {type(child).__name__} {tuple(child.weight.shape)}",
                          flush=True)
            else:
                _replace(child)

    _replace(model)
    return model.to(DEVICE)   # ← single .to() call moves everything uniformly


In [6]:
# ── Fine-tuning ───────────────────────────────────────────────────────────────
def finetune(model, train_loader, epochs, lr):
    if epochs == 0:
        return []
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler    = torch.amp.GradScaler(enabled=(DEVICE.type == "cuda"))
    epoch_accs = []
    print(f"  Fine-tuning {epochs} epochs @ lr={lr} ...", flush=True)
    for epoch in range(epochs):
        correct = total = 0
        for inputs, targets in train_loader:
            inputs  = inputs.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                loss = F.cross_entropy(model(inputs), targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            with torch.no_grad():
                preds    = model(inputs).argmax(1)
                correct += (preds == targets).sum().item()
                total   += targets.size(0)
        scheduler.step()
        acc = correct / total
        epoch_accs.append(round(acc, 6))
        print(f"    FT epoch {epoch+1}/{epochs}  train_acc={acc:.4f}", flush=True)
    return epoch_accs


In [7]:

# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*65}")
    print("  Weight Sharing — HashedNets")
    print(f"  Model  : {MODEL_ARCH}  |  BucketFraction: {BUCKET_FRACTION}")
    print(f"  Device : {DEVICE}  |  FineTune: {FINETUNE_EPOCHS} epochs")
    print(f"{'='*65}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
    baseline_acc = baseline["accuracy"]
    print(f"  Baseline accuracy: {baseline_acc:.4f}\n", flush=True)

    original_model   = load_model(ORIGINAL_MODEL_PATH, MODEL_ARCH)
    original_size_mb = model_size_mb(original_model)
    original_params  = param_count(original_model)
    print(f"  Original size  : {original_size_mb:.2f} MB  |  Params: {original_params:,}\n",
          flush=True)

    # ── Baseline FLOPs ────────────────────────────────────────────────────────
    print("  Measuring baseline GFLOPs ...", flush=True)
    baseline_gflops = compute_flops(original_model, DEVICE)
    print(f"  Baseline GFLOPs: {baseline_gflops:.4f}", flush=True)

    train_loader, test_loader = get_loaders()

    # ── Apply HashedNets ──────────────────────────────────────────────────────
    t0 = time.perf_counter()
    print("  Applying HashedNets ...", flush=True)
    hashed_model = apply_hashednets(original_model, BUCKET_FRACTION)
    compress_time = time.perf_counter() - t0
    print(f"  Done in {compress_time:.1f}s", flush=True)

    # Effective unique parameter count
    effective_params = count_hashed_params(hashed_model)
    print(f"  Effective unique params: {effective_params:,}  "
          f"({effective_params/original_params*100:.1f}% of original)", flush=True)

    # Pre-finetune eval
    pre_ft = evaluate(hashed_model, test_loader)
    print(f"  Pre-FT accuracy: {pre_ft['accuracy']:.4f}", flush=True)

    # Fine-tune
    ft_accs = finetune(hashed_model, train_loader, FINETUNE_EPOCHS, FINETUNE_LR)

    # Final eval & measurements
    metrics       = evaluate(hashed_model, test_loader)
    infer         = measure_inference(hashed_model, DEVICE)
    flops_g       = compute_flops(hashed_model, DEVICE)
    size_mb       = model_size_mb(hashed_model)

    acc_drop        = baseline_acc - metrics["accuracy"]
    param_reduction = 1.0 - effective_params / original_params

    print(f"\n  ✓ HashedNets Results:")
    print(f"    Accuracy          : {metrics['accuracy']:.4f}  (drop={acc_drop:+.4f})")
    print(f"    Size              : {size_mb:.2f} MB  (original={original_size_mb:.2f} MB)")
    print(f"    Param reduction   : {param_reduction*100:.1f}%")
    print(f"    FLOPs             : {flops_g:.4f} G")
    print(f"    Inference (single): {infer['single_img_gpu_ms']:.2f} ms")
    print(f"    Inference (batch) : {infer['batch128_gpu_ms']:.2f} ms  "
          f"({infer['throughput_imgs_sec']:.0f} img/s)")


    # ── Save model ────────────────────────────────────────────────────────────
    model_save_path = os.path.join(
        os.path.dirname(os.path.abspath(OUTPUT_JSON)),
        "__2__HashedNets.pth"
    )
    torch.save(hashed_model.state_dict(), model_save_path)
    print(f"  Saved model → {model_save_path}", flush=True)
    
    # ── Comparison block ──────────────────────────────────────────────────────
    comparison = {
        "accuracy":         {"baseline": round(baseline["accuracy"],   6),
                             "compressed": round(metrics["accuracy"],   6),
                             "delta": round(metrics["accuracy"] - baseline["accuracy"], 6)},
        "precision":        {"baseline": round(baseline.get("precision", 0), 6),
                             "compressed": round(metrics["precision"],  6),
                             "delta": round(metrics["precision"] - baseline.get("precision", 0), 6)},
        "recall":           {"baseline": round(baseline.get("recall", 0), 6),
                             "compressed": round(metrics["recall"],     6),
                             "delta": round(metrics["recall"] - baseline.get("recall", 0), 6)},
        "f1":               {"baseline": round(baseline.get("f1", 0),   6),
                             "compressed": round(metrics["f1"],         6),
                             "delta": round(metrics["f1"] - baseline.get("f1", 0), 6)},
        "num_parameters":   {"baseline": original_params,
                             "compressed": effective_params,
                             "delta": effective_params - original_params},
        "size_mb":          {"baseline": round(original_size_mb, 4),
                             "compressed": round(size_mb, 4),
                             "delta": round(size_mb - original_size_mb, 4)},
        "inference_ms":     {"baseline": baseline.get("inference_ms", {}),
                             "compressed": infer},
        "flops_G":          {"baseline": baseline_gflops,
                             "compressed": flops_g,
                             "delta": round(flops_g - baseline_gflops, 6)},
    }

    report = {
        "experiment"           : "weight_sharing_hashednets",
        "method"               : "weight_sharing",
        "variant"              : "HashedNets",
        "original_arch"        : MODEL_ARCH,
        "dataset"              : "CIFAR-10",
        "train_device"         : str(DEVICE),
        # hyperparameters
        "bucket_fraction"      : BUCKET_FRACTION,
        "hash_seed"            : HASH_SEED,
        "finetune_epochs"      : FINETUNE_EPOCHS,
        "finetune_lr"          : FINETUNE_LR,
        # description
        "method_description"   : (
            f"HashedNets: each weight index hashed to one of K={BUCKET_FRACTION*100:.0f}% buckets "
            f"via Knuth multiplicative hash h(i)=(SEED*i XOR i) mod K. "
            f"Shared parameter per bucket; gradients accumulated across all mapped weights."
        ),
        # timing
        "compression_time_s"   : round(compress_time, 2),
        # baseline
        "baseline"             : baseline,
        "original_size_mb"     : round(original_size_mb, 4),
        "original_params"      : original_params,
        "baseline_gflops"      : baseline_gflops,
        # performance
        "pre_finetune_accuracy": round(pre_ft["accuracy"], 6),
        "accuracy"             : round(metrics["accuracy"],  6),
        "precision"            : round(metrics["precision"], 6),
        "recall"               : round(metrics["recall"],    6),
        "f1"                   : round(metrics["f1"],        6),
        "accuracy_drop"        : round(acc_drop, 6),
        "finetune_epoch_accs"  : ft_accs,
        # size & efficiency
        "params"               : effective_params,
        "original_params"      : original_params,
        "param_reduction_pct"  : round(param_reduction * 100, 2),
        "size_mb"              : round(size_mb, 4),
        "original_size_mb"     : round(original_size_mb, 4),
        "compression_ratio"    : round(original_size_mb / size_mb, 4) if size_mb else None,
        "flops_G"              : flops_g,
        "inference_ms"         : infer,
        "saved_model_path"     : model_save_path,
        # ── Comparison metrics (before vs after) ──────────────────────────────
        "comparison"           : comparison,
        "status"               : "ok",
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n  ✓ Saved → {OUTPUT_JSON}")


if __name__ == "__main__":
    main()


  Weight Sharing — HashedNets
  Model  : resnet50  |  BucketFraction: 0.25
  Device : cuda  |  FineTune: 5 epochs

  Baseline accuracy: 0.9320

[model] Loading resnet50 from ../__2__baseline_resnet50_cifar10.pth ...
[model] Loaded. Params: 23,520,842
  Original size  : 90.03 MB  |  Params: 23,520,842

  Measuring baseline GFLOPs ...
  Baseline GFLOPs: 2.5957
  Applying HashedNets ...
    Hash: Conv2d (64, 3, 3, 3) → 432/1728 buckets (25.0%)
    Hash: Conv2d (64, 64, 1, 1) → 1024/4096 buckets (25.0%)
    Hash: Conv2d (64, 64, 3, 3) → 9216/36864 buckets (25.0%)
    Hash: Conv2d (256, 64, 1, 1) → 4096/16384 buckets (25.0%)
    Hash: Conv2d (256, 64, 1, 1) → 4096/16384 buckets (25.0%)
    Hash: Conv2d (64, 256, 1, 1) → 4096/16384 buckets (25.0%)
    Hash: Conv2d (64, 64, 3, 3) → 9216/36864 buckets (25.0%)
    Hash: Conv2d (256, 64, 1, 1) → 4096/16384 buckets (25.0%)
    Hash: Conv2d (64, 256, 1, 1) → 4096/16384 buckets (25.0%)
    Hash: Conv2d (64, 64, 3, 3) → 9216/36864 buckets (25.0%)
 